In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif, mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("train.csv")

In [3]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [5]:
df.shape

(1460, 81)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [7]:
cat_cols = df.select_dtypes(include='str').columns

cat_missing = df[cat_cols].isnull().sum()

cat_missing = cat_missing[cat_missing > 0]

print(cat_missing)

Alley           1369
MasVnrType       872
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64


In [8]:
df[cat_missing.index] = df[cat_missing.index].fillna('None')

In [9]:
df['PoolQC'].isnull().sum()

np.int64(0)

In [10]:
num_cols = df.select_dtypes(include=np.number).columns

num_missing = df[num_cols].isnull().sum()
num_missing = num_missing[num_missing > 0]

num_missing

LotFrontage    259
MasVnrArea       8
GarageYrBlt     81
dtype: int64

In [11]:
df[num_missing.index] = df[num_missing.index].fillna(df[num_missing.index].median())

In [12]:
df['LotFrontage'].isnull().sum()

np.int64(0)

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
print(df['SalePrice'].describe())

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64


In [15]:
df['PriceClass'] = (df['SalePrice'] > df['SalePrice'].median()).astype(int)

In [16]:
df['PriceClass'].value_counts()

PriceClass
0    732
1    728
Name: count, dtype: int64

In [17]:
df = df.drop(columns=['Id'])

In [18]:
X = df.drop(columns=['SalePrice', 'PriceClass'])
y_class = df['PriceClass']
y_reg = df['SalePrice']

In [19]:
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

In [20]:
print(X_train_class.shape)
print(X_test_class.shape)
print(y_train_class.shape)
print(y_test_class.shape)

(1168, 79)
(292, 79)
(1168,)
(292,)


In [21]:
X_encoded = pd.get_dummies(X, drop_first=True)
X_encoded.shape

(1460, 260)

In [22]:
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X_encoded, y_class, test_size=0.2, random_state=42, stratify=y_class
)
print(X_train_class.shape)
print(X_test_class.shape)

(1168, 260)
(292, 260)


In [23]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_encoded, y_reg, test_size=0.2, random_state=42
)
print(X_train_reg.shape)
print(X_test_reg.shape)

(1168, 260)
(292, 260)


In [24]:
clf_pipe = Pipeline([
    ('select', SelectKBest(score_func=mutual_info_classif, k=20)),
    ('scale', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

clf_params = {
    'select__k': [10, 20, 30, 'all'],
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance']
}

clf_grid = GridSearchCV(clf_pipe, clf_params, cv=5, scoring='accuracy', n_jobs=-1)
clf_grid.fit(X_train_class, y_train_class)

print(clf_grid.best_params_)
print(clf_grid.best_score_)

{'knn__n_neighbors': 11, 'knn__weights': 'distance', 'select__k': 30}
0.9143868530134626


In [25]:
y_pred_class = clf_grid.predict(X_test_class)
print("Test accuracy:", accuracy_score(y_test_class, y_pred_class))
print(classification_report(y_test_class, y_pred_class))

Test accuracy: 0.9041095890410958
              precision    recall  f1-score   support

           0       0.88      0.93      0.91       146
           1       0.93      0.88      0.90       146

    accuracy                           0.90       292
   macro avg       0.91      0.90      0.90       292
weighted avg       0.91      0.90      0.90       292



In [26]:
y_pred_class = clf_grid.predict(X_test_class)
print("Test accuracy:", accuracy_score(y_test_class, y_pred_class))
print(classification_report(y_test_class, y_pred_class))

Test accuracy: 0.9041095890410958
              precision    recall  f1-score   support

           0       0.88      0.93      0.91       146
           1       0.93      0.88      0.90       146

    accuracy                           0.90       292
   macro avg       0.91      0.90      0.90       292
weighted avg       0.91      0.90      0.90       292



In [27]:
reg_pipe = Pipeline([
    ('select', SelectKBest(score_func=mutual_info_regression, k=20)),
    ('scale', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

reg_params = {
    'select__k': [10, 20, 30, 'all'],
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance']
}

reg_grid = GridSearchCV(reg_pipe, reg_params, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
reg_grid.fit(X_train_reg, y_train_reg)

print(reg_grid.best_params_)
print(-reg_grid.best_score_)

{'knn__n_neighbors': 11, 'knn__weights': 'distance', 'select__k': 30}
33456.33765447088


In [31]:
y_pred_reg = reg_grid.predict(X_test_reg)
rmse = root_mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
print("Test RMSE:", rmse)
print("Test R2:", r2)

Test RMSE: 35770.53994139142
Test R2: 0.8331842097537191
